### Imports

In [1]:
import numpy as np
from maraboupy import Marabou
import time
import itertools

Instructions for updating:
non-resource variables are not supported in the long term


### Load Model

In [2]:
layers = '4'
width = '100'

In [3]:
model_path = f'../../models/{layers}_layers/model_{layers}x{width}.onnx'

In [4]:
display(Marabou.read_onnx(model_path))
display(len(Marabou.read_onnx(model_path).inputVars[0][0].tolist()))

108

- Get neural net feature indices and identify corresponding variables

In [ ]:
features = np.load("../../data/features.npy", allow_pickle=True)

print(features[:10], "...")
print()
print("size:", len(features))

['age' 'fnlwgt' 'education.num' 'capital.gain' 'capital.loss'
 'hours.per.week' 'workclass_ ?' 'workclass_ Federal-gov'
 'workclass_ Local-gov' 'workclass_ Never-worked'] ...

size: 108


In [ ]:
workclass = set(range(6,15))

education = set(range(15,31))

marital_status = set(range(31,38))

occupation = set(range(38,53))

relationship = set(range(53,59))

race = set(range(59,64))

sex = set(range(64,66))

native_country = set(range(66,108))

all_groups = [workclass, education, marital_status, occupation, relationship, race, sex, native_country]

### Fairness

- Before proceeding, it's important to understand why we need a fairness check for a model like this.
- Suppose a bank uses this model to decide who to grant loans to. If they make more than or equal to $50K, they are more likely to qualify for the loan, and vice versa.
- Now, what if the model learns that people belonging to a certain minority group are more likely to earn less than $50K? It'll give this information to the bank, which then is less likely to sponsor loans to these minority groups, perputuating the cycle. This is unfair.
- We thus want our model to be fair - regardless of changing certain identitarian criteria.

# DISCLAIMER

ALL THE CODE BELOW DOESN'T TECHINICALLY WORK.

----
- Define functions

In [ ]:
### [AI DISCLOSURE: Understanding original code and contraint-writing.]

def setup_networks(model_path:str, target_group:set[int], class_A:int, class_B:int):
    """
    Load two copies of the network and set up input equality constraints,
    except for the inputs for the one-hot-encoded class, where we enforce a reassignment.
    Returns the two network objects.
    """
    net1 = Marabou.read_onnx(model_path)
    net2 = Marabou.read_onnx(model_path)

    # Retrieve the input variable lists (assuming one input layer)
    inputs1 = net1.inputVars[0][0].tolist()
    inputs2 = net2.inputVars[0][0].tolist()

    # for both networks, add contraints so that multiclass one hot encoding is maintained for examples. 
    for group in all_groups:
        if group == target_group:
            continue

        group_vars1 = [inputs1[i] for i in group]
        group_vars2 = [inputs2[i] for i in group]
        coeffs = [1] * len(group)
        
        # Add constraint: sum(group_vars) = 1
        net1.addEquality(group_vars1, coeffs, 1) # For net1
        net2.addEquality(group_vars2, coeffs, 1) # For net2

    # For all inputs except those in the range of the one-hot-encoded classes, enforce equality: input1[i] == input2[i]
    for i in range(len(inputs1)):

        if i not in target_group:
            net1.addEquality([inputs1[i], inputs2[i]], [1, -1], 0)

    # CRITICAL FIX: Ensure exactly one race is selected in each network
    # For net1: class_A OR class_B is active (but not both)
    net1.addEquality([inputs1[class_A], inputs1[class_B]], [1, 1], 1)
    # A1 != B1
    
    # For net2: class_A OR class_B is active (but not both)  
    net2.addEquality([inputs2[class_A], inputs2[class_B]], [1, 1], 1)
    # A2 != B2

    # CRITICAL FIX: Swap the races between the two networks
    net1.addEquality([inputs1[class_A], inputs2[class_B]], [1, -1], 0)
    # A1 = B2 => 
    # A1 != A2, B1 != B2
    
    # make sure this applies to only (eg.) white and black persons in both networks.
    for i in (target_group - {class_A, class_B}):

        net1.setUpperBound(inputs1[i], 0) 
        net2.setUpperBound(inputs2[i], 0)
    
    return net1, net2

In [ ]:
### [AI DISCLOSURE: Understanding original code and some modifications necessary]

def check_case(case, model_path:str, target_group:set[int], class_A:int, class_B:int, threshold:float, epsilon:float):
    """
    Check one of the fairness violation cases.
    case == 1: net1 output >= threshold and net2 output < threshold
    case == 2: net1 output < threshold and net2 output >= threshold
    """
    start_time = time.time()
    

    #for class_A, class_B in list(itertools.combinations(target_group, 2)):
    # Setup networks and constraints common to both
    net1, net2 = setup_networks(model_path, target_group, class_A, class_B)
    
    # Retrieve the output variables
    output1 = int(net1.outputVars[0][0])
    output2 = int(net2.outputVars[0][0])

    
    if case == 1:
        net1.setLowerBound(output1, threshold)
        net1.setUpperBound(output2, threshold - epsilon)
    elif case == 2:
        net1.setUpperBound(output1, threshold - epsilon)
        net1.setLowerBound(output2, threshold)
    else:
        raise ValueError("Case must be 1 or 2.")
    
    exitCode, vals, stats = net1.solve()
    elapsed_time = time.time() - start_time
    
    return exitCode, vals, elapsed_time

In [ ]:
def fairness_check(model_path:str, target_group:set[int], all_groups:list[set[int]], 
                   threshold:float, epsilon:float):
    
    for class_A, class_B in list(itertools.combinations(target_group, 2)):
        
        exitCode1, vals1, time1 = check_case(1, model_path, target_group, class_A, class_B, threshold, epsilon)
        print("Time Taken (Case 1) =", round(time1, 4), "seconds")
        if exitCode1 == "sat":
            print(f"Fairness violation found (Case 1): Found a pair of classes ({class_A}, {class_B}) which change the outcome if swapped.")
            return vals1

        else:
            exitCode2, vals2, time2 = check_case(2, model_path, target_group, class_A, class_B, threshold, epsilon)
            print("Time Taken (Case 2) =", round(time2, 4), "seconds")
            
            if exitCode2 == "sat":
                print(f"Fairness violation found (Case 2): Found a pair of classes ({class_A}, {class_B}) which change the outcome if swapped.")
                return vals2

            else:
                pass
    
    print(f"No fairness violations were found for any of the classes in {target_group}")
    return None

#### Race

Let $race(x)$ denote the race one-hot subvector of $x$, and
let $RaceVals$ be the set of valid race encodings. For any $x \in \mathcal{X}$ define $x'$ to be $x$ with $race(x)$
replaced (all other coordinates unchanged). The model is fair at $x$ *iff*

$$\forall \ race(x) \in RaceVals \ \left( \^{y}(x) = \^{y}(x') \right)$$

**Verification target**: search for a violation—i.e., SAT of

$$\exists \ x \in \mathcal{X}, \ \exists \ race(x) \in RaceVals \ \left( \^{y}(x) \neq \^{y}(x') \right)$$



In [10]:
fairness_check(model_path, race, all_groups, 0.5, 1e-3)

/tmp/ipykernel_25699/920676000.py:18: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  output1 = int(net1.outputVars[0][0])
/tmp/ipykernel_25699/920676000.py:19: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  output2 = int(net2.outputVars[0][0])


Engine::processInputQuery: Input query (before preprocessing): 513 equations, 910 variables
unsat
Time Taken (Case 1) = 0.2482 seconds
Engine::processInputQuery: Input query (before preprocessing): 513 equations, 910 variables
unsat
Time Taken (Case 2) = 0.1159 seconds
Engine::processInputQuery: Input query (before preprocessing): 513 equations, 910 variables
unsat
Time Taken (Case 1) = 0.1537 seconds
Engine::processInputQuery: Input query (before preprocessing): 513 equations, 910 variables
unsat
Time Taken (Case 2) = 0.1268 seconds
Engine::processInputQuery: Input query (before preprocessing): 513 equations, 910 variables
unsat
Time Taken (Case 1) = 0.1459 seconds
Engine::processInputQuery: Input query (before preprocessing): 513 equations, 910 variables
unsat
Time Taken (Case 2) = 0.1468 seconds
Engine::processInputQuery: Input query (before preprocessing): 513 equations, 910 variables
unsat
Time Taken (Case 1) = 0.1556 seconds
Engine::processInputQuery: Input query (before preproce